# Module 1: Minimal RAG with LangChain

## Goal
In this notebook, we will build a very simple Retrieval-Augmented Generation (RAG) pipeline.

We will learn the following concepts step by step:

1. Create sample documents
2. Convert documents into chunks
3. Generate embeddings
4. Store them in a vector database
5. Retrieve relevant chunks for a question
6. Pass retrieved context to an LLM
7. Generate an answer

## Why this matters
This is the foundational workflow behind many enterprise RAG systems.

Later, we will improve this with:
- hybrid retrieval
- reranking
- citations
- evaluation
- observability
- workflow orchestration

## Architecture for this notebook

```text
Documents
  → Chunking
  → Embeddings
  → Vector Store
  → Retriever
  → LLM
  → Final Answer

In [1]:
# Install required packages
# Run this once in your notebook environment

%pip install -qU langchain langchain-openai langchain-chroma

Note: you may need to restart the kernel to use updated packages.


## Package Notes

We are using:

- `langchain` for the core framework
- `langchain-openai` for embeddings and chat model access
- `langchain-chroma` for the vector store

You will need an OpenAI API key for embeddings and generation.

In [1]:
import os
'''
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
'''

'\nimport getpass\n\nif not os.getenv("OPENAI_API_KEY"):\n    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")\n'

## Step 1: Create Sample Documents

To keep the first notebook simple, we will create a few in-memory documents manually.

Later, we will load:
- PDFs
- Word files
- HTML pages
- SharePoint / Confluence / blob storage content

For now, the goal is to understand the mechanics of RAG.

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
        LangChain is a framework for building LLM-powered applications.
        It helps developers work with prompts, models, retrievers, and chains.
        LangChain is often used to build chatbots and RAG applications.
        """,
        metadata={"source": "doc_1", "topic": "langchain"}
    ),
    Document(
        page_content="""
        LangGraph is used for orchestrating stateful workflows and agent systems.
        It is useful when applications require loops, branching, retries,
        or multi-step decision making.
        """,
        metadata={"source": "doc_2", "topic": "langgraph"}
    ),
    Document(
        page_content="""
        LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.
        """,
        metadata={"source": "doc_3", "topic": "langsmith"}
    ),
]

c:\Users\vammun01\OneDrive - Robert Half\repos\prod-rag\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
for i, doc in enumerate(documents, start=1):
    print(f"\n--- Document {i} ---")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content.strip())


--- Document 1 ---
Metadata: {'source': 'doc_1', 'topic': 'langchain'}
Content: LangChain is a framework for building LLM-powered applications.
        It helps developers work with prompts, models, retrievers, and chains.
        LangChain is often used to build chatbots and RAG applications.

--- Document 2 ---
Metadata: {'source': 'doc_2', 'topic': 'langgraph'}
Content: LangGraph is used for orchestrating stateful workflows and agent systems.
        It is useful when applications require loops, branching, retries,
        or multi-step decision making.

--- Document 3 ---
Metadata: {'source': 'doc_3', 'topic': 'langsmith'}
Content: LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.


## Step 2: Chunk the Documents

Why chunking?

We do not usually embed entire large documents because:
- retrieval becomes less precise
- embeddings become too broad
- irrelevant information gets mixed together

Chunking improves retrieval granularity.

In [4]:
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 6


In [6]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print("Content:", chunk.page_content.strip())


--- Chunk 1 ---
Metadata: {'source': 'doc_1', 'topic': 'langchain'}
Content: LangChain is a framework for building LLM-powered applications.
        It helps developers work with prompts, models, retrievers, and chains.

--- Chunk 2 ---
Metadata: {'source': 'doc_1', 'topic': 'langchain'}
Content: LangChain is often used to build chatbots and RAG applications.

--- Chunk 3 ---
Metadata: {'source': 'doc_2', 'topic': 'langgraph'}
Content: LangGraph is used for orchestrating stateful workflows and agent systems.
        It is useful when applications require loops, branching, retries,
        or multi-step decision making.

--- Chunk 4 ---
Metadata: {'source': 'doc_2', 'topic': 'langgraph'}
Content: or multi-step decision making.

--- Chunk 5 ---
Metadata: {'source': 'doc_3', 'topic': 'langsmith'}
Content: LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.


## Step 3: Create Embeddings and Store in Chroma

We now:
1. generate embeddings for each chunk
2. store them in a vector store

The vector store lets us search by semantic similarity.

In [7]:
%pip install -qU langchain-azure-ai azure-identity

Note: you may need to restart the kernel to use updated packages.


In [ ]:
'''
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="module1_rag_collection"
)
'''

In [29]:
import os
from dotenv import load_dotenv

# Load environment variables from a local .env file if available.
load_dotenv()

required = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME",
    "AZURE_OPENAI_ENDPOINT",
]
missing = [name for name in required if not (os.getenv(name) or "").strip()]

if missing:
    print("Missing env vars:", ", ".join(missing))
else:
    print("Azure OpenAI env vars are set.")
    print("Endpoint:", os.getenv("AZURE_OPENAI_ENDPOINT"))
    print("API version:", os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01 (default)")

Azure OpenAI env vars are set.
Endpoint: https://vm-demo-fdry-01.services.ai.azure.com/api/projects/proj01
API version: 2024-02-01


In [45]:
import os
from langchain_openai import AzureOpenAIEmbeddings, OpenAIEmbeddings

api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()

missing = []
if not api_key:
    missing.append("AZURE_OPENAI_API_KEY")
if not embedding_deployment:
    missing.append("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME")
if not endpoint:
    missing.append("AZURE_OPENAI_ENDPOINT")

if missing:
    raise ValueError(
        "Missing required environment variable(s): "
        + ", ".join(missing)
        + ". Set them in your terminal before launching Jupyter, or set os.environ[...] in a prior cell."
    )

# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_compat_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_compat_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized via Foundry OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

Embeddings client initialized via Foundry OpenAI-compatible endpoint.


In [33]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    collection_name="module1_rag_collection",
)

In [57]:
# Retrieve documents, metadata, and the raw embedding vectors
data = vectorstore._collection.get(include=['embeddings', 'documents', 'metadatas'])

# Access the first embedding in the store
first_embedding = data['embeddings'][0]
print(f"Vector Dimensions: {len(first_embedding)}")
print(f"Sample values: {first_embedding[:5]}")


Vector Dimensions: 1536
Sample values: [-0.00109931  0.00485348  0.04179017  0.00122494  0.05432722]


## Step 4: Create a Retriever

A retriever is the component that queries the vector store and returns the most relevant chunks.

In [34]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [40]:
question = "What is LangGraph used for?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Doc {i} ---")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content.strip())


--- Retrieved Doc 1 ---
Metadata: {'topic': 'langgraph', 'source': 'doc_2'}
Content: LangGraph is used for orchestrating stateful workflows and agent systems.
        It is useful when applications require loops, branching, retries,
        or multi-step decision making.

--- Retrieved Doc 2 ---
Metadata: {'topic': 'langchain', 'source': 'doc_1'}
Content: LangChain is often used to build chatbots and RAG applications.

--- Retrieved Doc 3 ---
Metadata: {'topic': 'langsmith', 'source': 'doc_3'}
Content: LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.


## Step 5: Send Retrieved Context to the LLM

Now we build the final RAG step:

- take the user question
- retrieve relevant chunks
- build a prompt
- ask the LLM to answer only from the provided context

In [51]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Ensure env vars are loaded even if setup cells were not run in order.
load_dotenv()

api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()

missing = []
if not api_key:
    missing.append("AZURE_OPENAI_API_KEY")
if not llm_deployment:
    missing.append("AZURE_OPENAI_LLM_DEPLOYMENT_NAME (or AZURE_OPENAI_CHAT_DEPLOYMENT_NAME)")
if not endpoint:
    missing.append("AZURE_OPENAI_ENDPOINT")

if missing:
    raise ValueError(
        "Missing required environment variable(s): "
        + ", ".join(missing)
        + ". Set them in your .env or in a prior cell via os.environ[...]"
    )

# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_compat_endpoint = f"{resource_root}/openai/v1"
    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_compat_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via Foundry OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question only using the context below.
If the answer is not in the context, say "I do not know based on the provided context."

Context:
{context}

Question:
{question}
""")

LLM initialized via Foundry OpenAI-compatible endpoint.


In [42]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\nContent: {doc.page_content}"
        for doc in docs
    )

context = format_docs(retrieved_docs)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(final_prompt.messages[0].content)


You are a helpful assistant.
Answer the question only using the context below.
If the answer is not in the context, say "I do not know based on the provided context."

Context:
Source: doc_2
Content: LangGraph is used for orchestrating stateful workflows and agent systems.
        It is useful when applications require loops, branching, retries,
        or multi-step decision making.

Source: doc_1
Content: LangChain is often used to build chatbots and RAG applications.

Source: doc_3
Content: LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.

Question:
What is LangGraph used for?



In [50]:
response = llm.invoke(final_prompt)
print(response.content)

LangGraph is used for orchestrating stateful workflows and agent systems, especially when applications require loops, branching, retries, or multi-step decision making.


## Step 6: Wrap This into a Reusable Function

This makes the notebook easier to test repeatedly.

In [55]:
def ask_rag(question: str, retriever, llm, prompt):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    response = llm.invoke(final_prompt)
    return {
        "question": question,
        "retrieved_docs": docs,
        "answer": response.content
    }

In [56]:
result = ask_rag(
    question="What does LangSmith help developers do?",
    retriever=retriever,
    llm=llm,
    prompt=prompt
)

print("Question:", result["question"])
print("\nAnswer:", result["answer"])

print("\nRetrieved Chunks:")
for i, doc in enumerate(result["retrieved_docs"], start=1):
    print(f"\nChunk {i}")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content.strip())

Question: What does LangSmith help developers do?

Answer: LangSmith helps developers with observability, debugging, tracing, and evaluation, so they can understand how their LLM applications behave in development and production.

Retrieved Chunks:

Chunk 1
Metadata: {'topic': 'langsmith', 'source': 'doc_3'}
Content: LangSmith is used for observability, debugging, tracing, and evaluation.
        It helps developers understand how their LLM applications behave in development
        and production.

Chunk 2
Metadata: {'source': 'doc_1', 'topic': 'langchain'}
Content: LangChain is a framework for building LLM-powered applications.
        It helps developers work with prompts, models, retrievers, and chains.

Chunk 3
Metadata: {'topic': 'langchain', 'source': 'doc_1'}
Content: LangChain is often used to build chatbots and RAG applications.


## What We Built

We built a minimal RAG pipeline with the following components:

- Documents
- Chunking
- Embeddings
- Vector store
- Retriever
- Prompt
- LLM answer generation

This is the base architecture for most RAG systems.

## What is missing for production?

This notebook is intentionally simple.

Production systems usually add:

- document loaders
- metadata filtering
- hybrid retrieval
- reranking
- source citations
- evaluation pipelines
- observability
- access control
- caching
- latency optimization

## Key Learning Takeaways

1. The vector store does not generate answers.
   It only helps retrieve relevant chunks.

2. The LLM does not search the knowledge base directly.
   It answers using the retrieved context.

3. Retrieval quality strongly affects answer quality.

4. Chunking strategy is important.

5. This notebook is the foundation for enterprise RAG.